# Construcción de df_pacientes_trayectorias
Este notebook construye la base `df_pacientes_trayectorias` a partir de `df_base_limpia` (episodios) y `df_traslados_strict` (transiciones).
Cada fila representará **un paciente con su trayectoria hospitalaria**.

Se implementarán y compararán dos metodologías:
1. **Metodología 1 — Anclada en Traslados (Principal)**
2. **Metodología 2 — Desde Episodios (Alternativa)**


In [21]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import pandas as pd
import numpy as np
from src.config import *

# Carga de datos
df_base = pd.read_excel("../data/final_data/df_base_limpia.xlsx")
# df_traslados = pd.read_excel("../data/final_data/df_traslados_strict.xlsx")

# Ordenar
df_base['fecha_ingreso'] = pd.to_datetime(df_base['fecha_ingreso'])
df_base['fecha_egreso'] = pd.to_datetime(df_base['fecha_egreso'])
df_base = df_base.sort_values(['paciente_id', 'fecha_ingreso']).copy()

# df_traslados['fecha_egreso_origen'] = pd.to_datetime(df_traslados['fecha_egreso_origen'])
# df_traslados['fecha_ingreso_destino'] = pd.to_datetime(df_traslados['fecha_ingreso_destino'])


## 1. Limpieza (nivel paciente — filtros estrictos)
Aplicamos filtros a `df_base` para obtener una lista de `pacientes_validos`.

In [22]:
# --- fechas incompletas
df_base['flag_fechas_incompletas'] = df_base['fecha_ingreso'].isna() | df_base['fecha_egreso'].isna()

# --- consistencia temporal (tolerancia de 10 mins)
TOL_MINUTOS = 10
df_base['delta_min'] = (df_base['fecha_egreso'] - df_base['fecha_ingreso']).dt.total_seconds() / 60
df_base['flag_fechas_invalidas'] = df_base['delta_min'] < -TOL_MINUTOS

# --- consistencia de edad (delta > 2)
df_base['edad_next'] = df_base.groupby('paciente_id')['edad'].shift(-1)
df_base['delta_edad_episodios'] = df_base['edad_next'] - df_base['edad']
df_base['flag_edad_inconsistente'] = df_base['delta_edad_episodios'].abs() > 2

# --- egreso administrativo final
idx_last = df_base.groupby('paciente_id')['fecha_ingreso'].idxmax()
df_last = df_base.loc[idx_last, ['paciente_id', 'tipo_egreso']].copy()
df_last['flag_fin_admin'] = df_last['tipo_egreso'] == 'administrativo'

# --- agregación por paciente
df_flags_paciente = df_base.groupby('paciente_id').agg({
    'flag_fechas_incompletas': 'max',
    'flag_fechas_invalidas': 'max',
    'flag_edad_inconsistente': 'max'
}).reset_index()

df_flags_paciente = df_flags_paciente.merge(df_last[['paciente_id', 'flag_fin_admin']], on='paciente_id', how='left')

# paciente válido si no incumple ninguna regla
df_flags_paciente['paciente_valido'] = (
    ~df_flags_paciente['flag_fechas_incompletas'] &
    ~df_flags_paciente['flag_fechas_invalidas'] &
    ~df_flags_paciente['flag_edad_inconsistente'] &
    ~df_flags_paciente['flag_fin_admin']
)

pacientes_validos = df_flags_paciente.loc[df_flags_paciente['paciente_valido'], 'paciente_id']
df_base_filtrada = df_base[df_base['paciente_id'].isin(pacientes_validos)].copy()

print(f"Total pacientes originales: {df_base['paciente_id'].nunique()}")
print(f"Total pacientes válidos: {len(pacientes_validos)}")

# =============================================================================
# CREACIÓN DE EPISODIOS CLÍNICOS para un  mismo id (Cortes por reingreso)
# =============================================================================
UMBRAL_CORTE_DIAS = 14 # Si pasan más de 14 días entre el alta y el nuevo ingreso, es otro episodio

# 1. Ordenar estrictamente por tiempo
df_base_filtrada = df_base_filtrada.sort_values(['paciente_id', 'fecha_ingreso'])

# 2. Calcular cuántos días pasaron desde su ÚLTIMA alta
df_base_filtrada['dias_desde_ultimo_egreso'] = (
    df_base_filtrada['fecha_ingreso'] - df_base_filtrada.groupby('paciente_id')['fecha_egreso'].shift(1)
).dt.total_seconds() / (3600*24)

# 3. Marcar dónde empieza un nuevo episodio (o si es el primero del paciente)
df_base_filtrada['es_nuevo_episodio'] = (
    (df_base_filtrada['dias_desde_ultimo_egreso'] > UMBRAL_CORTE_DIAS) | 
    (df_base_filtrada['dias_desde_ultimo_egreso'].isna())
)

# 4. Asignar un número de episodio acumulativo por paciente
df_base_filtrada['num_episodio'] = df_base_filtrada.groupby('paciente_id')['es_nuevo_episodio'].cumsum()

# 5. CREAR LA NUEVA LLAVE PRIMARIA: paciente_id + "_" + num_episodio
df_base_filtrada['paciente_episodio_id'] = df_base_filtrada['paciente_id'].astype(str) + "_E" + df_base_filtrada['num_episodio'].astype(str)

print(f"Total pacientes únicos: {df_base_filtrada['paciente_id'].nunique()}")
print(f"Total episodios clínicos distintos: {df_base_filtrada['paciente_episodio_id'].nunique()}")


Total pacientes originales: 22729
Total pacientes válidos: 19980
Total pacientes únicos: 19980
Total episodios clínicos distintos: 19985


# METODOLOGÍA sandwich

In [23]:
# METODOLOGÍA — LÓGICA SÁNDWICH (Desde df_base_filtrada)
# =============================================================================

def construir_trayectoria_sandwich(grp):
    fecha_ingreso_paciente = grp['fecha_ingreso'].min()
    
    # 1. Separar Relleno y Pan de Abajo (Buscamos EXACTAMENTE los traslados de red)
    mask_traslado = grp['tipo_egreso'].astype(str).str.lower() == 'traslado-hospital-de-la-red'
    
    df_traslados = grp[mask_traslado].sort_values('fecha_ingreso')
    df_finales = grp[~mask_traslado].sort_values('fecha_ingreso')
    
    flags = {'sin_evento_final': 0, 'multiples_finales': 0, 'cierre_administrativo': 0}
    
    # 2. JERARQUÍA DEL DESENLACE (La Muerte mata todo lo demás)
    if len(df_finales) == 0:
        flags['sin_evento_final'] = 1
        fecha_egreso_paciente = grp['fecha_egreso'].max()
        desenlace = 'solo_traslados'
    else:
        if len(df_finales) > 1:
            flags['multiples_finales'] = 1
            
        # LA REGLA DE ORO: Si 'muerte' está en los registros, ese es el final absoluto.
        if (df_finales['tipo_egreso'].astype(str).str.lower() == 'muerte').any():
            desenlace = 'muerte'
            # Tomamos la fecha de la muerte real, no del papel administrativo posterior
            fecha_egreso_paciente = df_finales[df_finales['tipo_egreso'].astype(str).str.lower() == 'muerte']['fecha_egreso'].iloc[0]
        else:
            fecha_egreso_paciente = df_finales.iloc[-1]['fecha_egreso']
            desenlace = df_finales.iloc[-1]['tipo_egreso']
            
        # Marcamos si es administrativo para borrarlo luego
        if 'administrativo' in str(desenlace).lower():
            flags['cierre_administrativo'] = 1

    # 3. ARMAR LAS RUTAS (Cruda vs Limpia)
    hospitales_seq = df_traslados['hospital_origen'].tolist() + df_finales['hospital_origen'].tolist()
    
    # Ruta Colapsada (Para armar la red de hospitales)
    hospitales_limpios = []
    if hospitales_seq:
        hospitales_limpios = [hospitales_seq[0]]
        for h in hospitales_seq[1:]:
            if h != hospitales_limpios[-1]:
                hospitales_limpios.append(h)
                
    duracion_dias = (fecha_egreso_paciente - fecha_ingreso_paciente).total_seconds() / (3600*24)
    
    # Calculamos cuántas veces se movió adentro de un mismo hospital
    movimientos_internos = len(hospitales_seq) - len(hospitales_limpios)
                
    return pd.Series({
        'paciente_id_original': grp['paciente_id'].iloc[0], # Guardamos el DNI real por si acaso
        'lista_hospitales': hospitales_limpios,
        'trayectoria_hospitalaria': str(hospitales_limpios),
        'hospital_inicio': hospitales_limpios[0] if hospitales_limpios else None,
        'hospital_final': hospitales_limpios[-1] if hospitales_limpios else None,
        'fecha_ingreso_paciente': fecha_ingreso_paciente,
        'fecha_egreso_paciente': fecha_egreso_paciente,
        'duracion_total_dias': duracion_dias,
        'movimientos_internos_ocultos': movimientos_internos, # ¡Visibilidad recuperada!
        'desenlace': desenlace,
        'flag_sin_evento_final': flags['sin_evento_final'],
        'flag_multiples_finales': flags['multiples_finales'],
        'flag_cierre_admin': flags['cierre_administrativo']
    })

# ¡ATENCIÓN! Ahora agrupamos por el Episodio, no por el paciente
df_trayectorias_limpias = df_base_filtrada.groupby('paciente_episodio_id').apply(construir_trayectoria_sandwich).reset_index()
df_trayectorias_final.to_excel("../data/final_data/df_trayectorias_final.xlsx", index=False)


In [ ]:
# # DERIVACIÓN DE TRASLADOS DESDE LA TRAYECTORIA
# # =============================================================================

# registros_traslados = []

# for _, row in df_trayectorias_limpias.iterrows():
#     episodio_id = row['paciente_episodio_id']
#     paciente_real = row['paciente_id_original']
#     hospitales = row['lista_hospitales']
    
#     if len(hospitales) > 1:
#         # Extraemos solo los episodios de ESTE paciente_episodio para buscar las fechas exactas
#         historial_paciente = df_base_filtrada[df_base_filtrada['paciente_episodio_id'] == episodio_id]
        
#         for i in range(len(hospitales) - 1):
#             h_origen = hospitales[i]
#             h_destino = hospitales[i+1]
            
#             # Buscar la fecha de egreso del origen y la fecha de ingreso del destino
#             # (Agarramos la última ocurrencia del origen antes de saltar, y la primera del destino al llegar)
#             fecha_out = historial_paciente[historial_paciente['hospital_origen'] == h_origen]['fecha_egreso'].max()
#             fecha_in = historial_paciente[historial_paciente['hospital_origen'] == h_destino]['fecha_ingreso'].min()
            
#             delta_traslado_horas = (fecha_in - fecha_out).total_seconds() / 3600
            
#             registros_traslados.append({
#                 'paciente_episodio_id': episodio_id,
#                 'paciente_id_real': paciente_real,
#                 'hospital_origen': h_origen,
#                 'hospital_destino': h_destino,
#                 'paso_trayectoria': i + 1,
#                 'demora_traslado_horas': delta_traslado_horas,
#                 # Flag Zombie: Si tardó más de 48 horas en ir de un hospital a otro, algo raro pasó
#                 'flag_zombie': 1 if (pd.notna(delta_traslado_horas) and delta_traslado_horas > 48) else 0,
#                 # Flag Cronología Rota: Si llegó antes de salir (error de tipeo del hospital)
#                 'flag_fecha_negativa': 1 if (pd.notna(delta_traslado_horas) and delta_traslado_horas < 0) else 0
#             })

# df_traslados_derivados = pd.DataFrame(registros_traslados)
# df_trayectorias_final = df_trayectorias_limpias.drop(columns=['lista_hospitales'])


# # Guardamos los resultados
# df_trayectorias_final.to_excel("../data/final_data/df_pacientes_trayectorias_oficial.xlsx", index=False)
# df_traslados_derivados.to_excel("../data/final_data/df_traslados_derivados.xlsx", index=False)

# print(f"Trayectorias construidas: {len(df_trayectorias_final)}")
# print(f"Traslados reales derivados: {len(df_traslados_derivados)}")

Trayectorias construidas: 19985
Traslados reales derivados: 2019


In [ ]:
# # MÓDULO DE CONTROL DE CALIDAD (QA)
# # =============================================================================

# print("INICIANDO PRUEBAS DE CALIDAD...")
# print("="*50)

# # Cargar las bases finales (asumiendo que ya están en memoria o las lees del excel)
# df_tray = df_trayectorias_final
# df_tras = df_traslados_derivados

# # ---------------------------------------------------------
# # TEST 1: SALUD DE LAS TRAYECTORIAS (BASE 3)
# # ---------------------------------------------------------
# print("\n[TEST 1] SALUD DE TRAYECTORIAS")
# total_pacientes = len(df_tray)

# # 1.1 Coherencia de Desenlaces
# casos_ideales = df_tray[(df_tray['flag_sin_evento_final'] == 0) & (df_tray['flag_multiples_finales'] == 0)]
# pct_ideales = (len(casos_ideales) / total_pacientes) * 100

# casos_solo_traslados = df_tray['flag_sin_evento_final'].sum()
# casos_multiples = df_tray['flag_multiples_finales'].sum()

# print(f"✔️ Pacientes con 1 evento final claro: {len(casos_ideales)} ({pct_ideales:.1f}%)")
# print(f"⚠️ Pacientes con múltiples finales (Ambigüedad): {casos_multiples}")
# print(f"🚨 Pacientes iterando sin alta (Solo traslados): {casos_solo_traslados}")

# # 1.2 Coherencia Temporal (Física)
# duracion_negativa = df_tray[df_tray['duracion_total_dias'] < 0]
# print(f"⏱️ Pacientes con duración negativa (Error grave de fechas): {len(duracion_negativa)}")

# # ---------------------------------------------------------
# # TEST 2: INTEGRIDAD RELACIONAL (TRAYECTORIAS vs TRASLADOS)
# # ---------------------------------------------------------
# print("\n[TEST 2] INTEGRIDAD MATEMÁTICA (TRAYECTORIAS <-> TRASLADOS)")

# # 2.1 Coincidencia de Pacientes
# pacientes_con_traslados_esperados = df_tray[df_tray['trayectoria_hospitalaria'].str.contains(',')]['paciente_id'].nunique()
# pacientes_en_base_traslados = df_tras['paciente_id'].nunique()

# if pacientes_con_traslados_esperados == pacientes_en_base_traslados:
#     print(f"✔️ Match perfecto: {pacientes_en_base_traslados} pacientes tienen traslados en ambas bases.")
# else:
#     print(f"🚨 ERROR: Discrepancia en pacientes conectados. Esperados: {pacientes_con_traslados_esperados}, Encontrados: {pacientes_en_base_traslados}")

# # ---------------------------------------------------------
# # TEST 3: SALUD DE LOS TRASLADOS (BASE 2)
# # ---------------------------------------------------------
# print("\n[TEST 3] ANÁLISIS DE LA RED DE TRASLADOS")

# total_traslados = len(df_tras)

# # 3.1 Nodos idénticos (A -> A)
# # Esto no debería ocurrir gracias a la limpieza `if h != hospitales_limpios[-1]`
# bucles_falsos = df_tras[df_tras['hospital_origen'] == df_tras['hospital_destino']]
# if len(bucles_falsos) == 0:
#     print("✔️ Cero bucles falsos (A -> A) detectados. Limpieza exitosa.")
# else:
#     print(f"🚨 ERROR: Se encontraron {len(bucles_falsos)} traslados donde Origen == Destino.")

# # 3.2 Longitud de las cadenas
# max_pasos = df_tras['paso_trayectoria'].max()
# print(f"🔗 Longitud máxima de una cadena de traslados: {max_pasos} pasos.")

# print("="*50)
# print("FIN DE PRUEBAS.")

INICIANDO PRUEBAS DE CALIDAD...

[TEST 1] SALUD DE TRAYECTORIAS
✔️ Pacientes con 1 evento final claro: 18143 (90.8%)
⚠️ Pacientes con múltiples finales (Ambigüedad): 1837
🚨 Pacientes iterando sin alta (Solo traslados): 0
⏱️ Pacientes con duración negativa (Error grave de fechas): 0

[TEST 2] INTEGRIDAD MATEMÁTICA (TRAYECTORIAS <-> TRASLADOS)
✔️ Match perfecto: 1832 pacientes tienen traslados en ambas bases.

[TEST 3] ANÁLISIS DE LA RED DE TRASLADOS
✔️ Cero bucles falsos (A -> A) detectados. Limpieza exitosa.
🔗 Longitud máxima de una cadena de traslados: 8 pasos.
FIN DE PRUEBAS.
